# **Séance 5 — Lire et enregistrer des données : JSON**

*Master 2 TNAH, École nationale des chartes — Introduction au langage de programmation Python*

---

### **Objectifs de la séance**

- comprendre et construire des **décorateurs** ;
- connaître le format **JSON** et ses correspondances avec les types Python ;
- **lire et enregistrer** des fichiers avec `open()`, la bibliothèque `json` et `pathlib` ;
- **parcourir** un fichier JSON produit par une institution : le manifeste IIIF de Berne, Burgerbibliothek, Cod. 318.

### **Sommaire**

1. Les décorateurs
2. Le format JSON
3. Lire et enregistrer des fichiers
4. Parcourir un manifeste IIIF

---

### **Données du corpus**

Comme à chaque séance, exécutez d'abord la cellule ci-dessous. Elle recrée les données des séances précédentes. Relancez-la si vous redémarrez le noyau.

*La classe `Manuscrit` des séances 3 et 4 appelait `bestiaire.siecle()`, depuis le module de la séance 2. Ici, elle appelle directement la fonction `siecle()` définie dans la même cellule : le module n'est pas nécessaire.*

In [ ]:
import json
from pathlib import Path

# Séance 1
titre = "III, 14." + " " + "De la colombe."
creatures = ["dragon", "licorne", "lion", "panthère", "phénix"]

manuscrit = {
    "cote": "MS 24",
    "bibliotheque": "Aberdeen, University Library",
    "date": 1200,
    "numerise": True
}

manuscrit_oxford = {
    "cote": "MS. Ashmole 1511",
    "bibliotheque": "Oxford, Bodleian Library",
    "date": 1210,
    "numerise": True
}

corpus = [manuscrit, manuscrit_oxford]


# Séance 4
class DateInvalide(Exception):
    """Erreur levée quand une date n'est pas un nombre entier."""
    pass


def siecle(date: int) -> str:
    """Renvoyer le siècle d'une date."""
    if not isinstance(date, int):
        raise DateInvalide("la date doit être un nombre entier")
    if date <= 1200:
        resultat = "XIIe siècle"
    else:
        resultat = "XIIIe siècle"
    return resultat


# Séances 3 et 4
class Manuscrit:
    """Un manuscrit du corpus."""
    genre = "bestiaire"

    def __init__(self, cote, bibliotheque, date):
        self.cote = cote
        self.bibliotheque = bibliotheque
        self.date = date
        self.creatures = []

    def cote_complete(self):
        """Renvoyer la bibliothèque et la cote du manuscrit."""
        return f"{self.bibliotheque}, {self.cote}"

    def siecle(self):
        """Renvoyer le siècle du manuscrit."""
        return siecle(self.date)

    def ajouter_creature(self, nom):
        """Ajouter une créature à la liste du manuscrit."""
        self.creatures.append(nom)

    def fiche(self):
        """Renvoyer la fiche du manuscrit."""
        return (f"{self.cote_complete()} ({self.genre})\n"
                f"  Siècle : {self.siecle()}\n"
                f"  Créatures : {self.creatures}")


print("Données chargées.")

---

## **1. Les décorateurs**

### **1.1 Rappel : donner une fonction à une autre fonction**

Avec des parenthèses, on **appelle** une fonction et on obtient son résultat :

In [ ]:
siecle(1210)

Sans parenthèses, on désigne **la fonction elle-même**, comme une valeur. Jupyter affiche alors une description de l'objet :

In [ ]:
siecle

Comme toute valeur, une fonction peut être rangée dans une variable. On ne l'exécute pas : on lui donne un second nom.

In [ ]:
f = siecle      # pas de parenthèses : on ne l'exécute pas
f(1210)

En séance 2, vous avez déjà donné une fonction à une autre fonction, sans l'appeler : avec `sorted()` et son paramètre `key`. C'est `sorted()` qui appelle la fonction, pour chaque manuscrit.

In [ ]:
corpus_melange = [manuscrit_oxford, manuscrit]

for notice in sorted(corpus_melange, key=lambda manuscrit: manuscrit["date"]):
    print(notice["date"], notice["cote"])

#### **Exercice 1 : Trier les créatures par longueur**

1. Triez la liste `creatures` de la plus courte à la plus longue, en donnant la fonction `len` au paramètre `key` de `sorted()`.
2. Que se passe-t-il si vous écrivez `key=len()` ? Pourquoi ?

*Solution :*

In [ ]:
# Votre code ici

### **1.2 Construire un décorateur**

**L'objectif** : afficher un message avant et après une fonction, **sans toucher à son code**.

**Une image : la reliure.** On ne réécrit pas les feuillets d'un manuscrit pour le relier : on les place **à l'intérieur** d'une reliure. Un décorateur fait la même chose : il place la fonction d'origine à l'intérieur d'une nouvelle fonction, qui ajoute quelque chose avant et après.

Un **décorateur** est une fonction qui :

1. **reçoit** une fonction, sans parenthèses ;
2. **fabrique** une nouvelle fonction, qui affiche un message, appelle la fonction reçue, puis affiche un second message ;
3. **renvoie** cette nouvelle fonction, sans l'exécuter.

In [ ]:
def annoncer(fonction):                       # 1. reçoit la fonction
    def nouvelle_fonction():                  # 2. fabrique la reliure
        print("Avant l'exécution de la fonction")
        resultat = fonction()                 #    appelle la fonction d'origine
        print("Après l'exécution de la fonction")
        return resultat
    return nouvelle_fonction                  # 3. renvoie la reliure

À ce stade, **rien ne s'affiche** : on a seulement défini `annoncer`.

### **1.3 Appliquer un décorateur**

Voici la fonction à décorer, avant toute décoration :

In [ ]:
def afficher_titre():
    print(titre)


afficher_titre()

**À la main.** La ligne suivante se lit de droite à gauche :

1. on donne la fonction `afficher_titre` à `annoncer` ;
2. `annoncer` renvoie `nouvelle_fonction` ;
3. on la range sous le nom `afficher_titre`.

Désormais, `afficher_titre` désigne la fonction **reliée**.

In [ ]:
afficher_titre = annoncer(afficher_titre)

afficher_titre()

À l'appel :

1. on appelle `afficher_titre`, qui est maintenant `nouvelle_fonction` ;
2. elle affiche « Avant… » ;
3. elle appelle `fonction()`, c'est-à-dire l'ancienne `afficher_titre`, qui affiche le titre ;
4. elle affiche « Après… ».

**Avec `@`.** `@annoncer` est une **abréviation** de `afficher_titre = annoncer(afficher_titre)` :

In [ ]:
@annoncer
def afficher_titre():
    print(titre)


afficher_titre()

#### **Exercice 2 : Prédire l'affichage**

Sans exécuter la cellule ci-dessous, écrivez dans la cellule de réponse ce qu'elle va afficher, ligne par ligne. Exécutez-la ensuite pour vérifier.

```python
@annoncer
def afficher_creatures():
    print(creatures)


print("Début")
afficher_creatures()
print("Fin")
```

*Votre réponse :*

In [ ]:
@annoncer
def afficher_creatures():
    print(creatures)


print("Début")
afficher_creatures()
print("Fin")

### **1.4 Décorer une fonction qui a des paramètres**

Essayons de décorer `siecle()`. Pour garder la fonction d'origine intacte dans la suite du notebook, on range la version décorée sous un autre nom, `siecle_annoncee`.

Décommentez les lignes pour voir l'erreur.

In [ ]:
# siecle_annoncee = annoncer(siecle)
# siecle_annoncee(1210)
# TypeError: annoncer.<locals>.nouvelle_fonction() takes 0 positional arguments but 1 was given

`siecle_annoncee` est `nouvelle_fonction`, qui n'accepte **aucun argument** : la reliure n'a pas prévu de place pour `1210`. Il faut que `nouvelle_fonction` accepte **n'importe quels arguments** et les transmette à la fonction d'origine.

**Rappel de la séance 2** : `*args` regroupe les arguments **positionnels** dans un **tuple**, et `**kwargs` regroupe les arguments **nommés** dans un **dictionnaire**.

In [ ]:
def decrire_creatures(cote, *noms):
    return f"{cote} : {len(noms)} créatures {noms}"


def notice_libre(**infos):
    return infos


print(decrire_creatures("MS 24", "dragon", "licorne", "phénix"))
print(notice_libre(cote="MS 24", date=1200, numerise=True))

Dans un décorateur :

- dans le `def`, les étoiles **regroupent** tous les arguments reçus, quels qu'ils soient ;
- à l'appel de `fonction`, elles les **déballent** pour les transmettre tels quels.

In [ ]:
def annoncer(fonction):
    def nouvelle_fonction(*args, **kwargs):     # accepte tout
        print("Avant l'exécution de la fonction")
        resultat = fonction(*args, **kwargs)    # transmet tout
        print("Après l'exécution de la fonction")
        return resultat
    return nouvelle_fonction


siecle_annoncee = annoncer(siecle)
print(siecle_annoncee(1210))

« XIIIe siècle » s'affiche **en dernier** : `print()` attend que `siecle_annoncee(1210)` ait **renvoyé** son résultat, et le décorateur a donc déjà tout affiché.

`return resultat` est indispensable : c'est lui qui renvoie le résultat de `siecle()`. Sans cette ligne, `print()` afficherait `None`.

Le même décorateur fonctionne aussi avec une fonction sans paramètre :

In [ ]:
@annoncer
def afficher_titre():
    print(titre)


afficher_titre()

#### **Erreurs fréquentes**

On utilise une fonction neuve, `afficher_cote()`, pour chacun des trois pièges.

**Piège 1 : des parenthèses en trop.** `annoncer(afficher_cote())` **appelle** `afficher_cote` tout de suite : la cote s'affiche, et `annoncer` reçoit son résultat, `None`, au lieu de la fonction. Décommentez la dernière ligne pour voir l'erreur.

In [ ]:
def afficher_cote():
    print(manuscrit["cote"])


decoree = annoncer(afficher_cote())   # la cote s'affiche dès maintenant
# decoree()
# Avant l'exécution de la fonction
# TypeError: 'NoneType' object is not callable

**Piège 2 : oublier `return nouvelle_fonction`.** Le décorateur ne renvoie rien : le nom décoré vaut `None`, et l'appel provoque une erreur.

In [ ]:
def annoncer_sans_return(fonction):
    def nouvelle_fonction(*args, **kwargs):
        print("Avant l'exécution de la fonction")
        resultat = fonction(*args, **kwargs)
        print("Après l'exécution de la fonction")
        return resultat
    # return nouvelle_fonction oublié


@annoncer_sans_return
def afficher_cote():
    print(manuscrit["cote"])


print(afficher_cote)
# afficher_cote()
# TypeError: 'NoneType' object is not callable

**Piège 3 : oublier `return resultat`.** La fonction décorée s'exécute bien, mais son résultat est perdu.

In [ ]:
def annoncer_sans_resultat(fonction):
    def nouvelle_fonction(*args, **kwargs):
        print("Avant l'exécution de la fonction")
        fonction(*args, **kwargs)              # le résultat n'est pas gardé
        print("Après l'exécution de la fonction")
    return nouvelle_fonction


siecle_perdu = annoncer_sans_resultat(siecle)
print(siecle_perdu(1210))

#### **Exercice 3 : Un décorateur qui signale le calcul**

1. Créez un décorateur `signaler` qui affiche « Calcul en cours… » avant l'appel de la fonction et « Calcul terminé. » après.
2. Appliquez-le **avec `@`** à une nouvelle fonction `nombre_creatures(liste)`, qui renvoie le nombre d'éléments de la liste. Testez-la avec `creatures`.
3. Appliquez-le **à la main** à `siecle()`, sous le nom `siecle_signalee`. Testez-la avec `1210`.

*Solution :*

In [ ]:
# Votre code ici

#### **Exercice 4 (facultatif) : Vérifier la date avec un décorateur**

1. Créez un décorateur `verifier_date` qui lève `DateInvalide` si le **premier argument** de la fonction n'est pas un entier, et qui appelle la fonction sinon.
2. Appliquez-le avec `@` à une version de `siecle()` **sans** `isinstance()` : la vérification est faite par le décorateur.
3. Testez avec `1210`, puis avec `"vers 1210"` dans un `try` / `except`.

*Indice : dans `nouvelle_fonction(*args, **kwargs)`, le premier argument est `args[0]`.*

*Solution :*

In [ ]:
# Votre code ici

### **1.5 Pourquoi utiliser des décorateurs ?**

**Séparer les tâches**

- La fonction fait son travail : `siecle()` calcule un siècle.
- Le décorateur se charge d'une tâche annexe : annoncer l'appel, vérifier les arguments, gérer les erreurs.
- Chaque partie a un seul rôle : le code est plus lisible.

**Ne pas modifier ce qui fonctionne**

- La fonction reste telle qu'elle a été écrite et testée : on ajoute, on ne modifie pas.

**Réutiliser le code**

- Un même décorateur s'applique à plusieurs fonctions, sans dupliquer le code.
- Pour changer le comportement ajouté, on ne modifie que le décorateur : toutes les fonctions décorées en profitent.

---

## **2. Le format JSON**

### **2.1 Pourquoi enregistrer ses données ?**

Depuis la séance 1, nous décrivons les manuscrits du corpus. Mais à chaque séance, il a fallu relancer une cellule qui recrée toutes les données : les variables n'existent que **pendant l'exécution** du programme. Si l'on ferme le notebook, `corpus` disparaît.

**Une image : le catalogue.**

- Une **variable**, c'est une notice qu'on garde en tête : elle disparaît quand on referme le notebook.
- Un **fichier**, c'est une notice inscrite au catalogue : on la retrouve plus tard, et d'autres peuvent la consulter.

Deux formats ce semestre : **JSON** (aujourd'hui), pour des données **imbriquées**, et **CSV** (séance 6), pour des **tableaux**.

### **2.2 La syntaxe JSON**

JSON (*JavaScript Object Notation*) est un format **texte**, léger et lisible, qui ne dépend d'aucun langage. Il se compose :

- d'**objets** `{ }`, faits de paires **clé-valeur**, dont les clés sont toujours des chaînes ;
- de **listes** `[ ]`, faites de valeurs ordonnées.

Les valeurs peuvent être des chaînes (`"MS 24"`), des nombres (`1200`), des booléens (`true` / `false`), la valeur vide `null`, ou des listes et des objets imbriqués.

**Attention** : les chaînes s'écrivent **toujours entre guillemets doubles**.

La notice de MS 24 en JSON :

```json
{
  "cote": "MS 24",
  "bibliotheque": "Aberdeen, University Library",
  "date": 1200,
  "numerise": true,
  "nb_folios": null,
  "creatures": ["dragon", "licorne", "lion", "panthère", "phénix"]
}
```

### **2.3 De Python à JSON, et retour**

- **Sérialiser** : transformer un objet Python en texte JSON, pour l'enregistrer.
- **Désérialiser** : relire du texte JSON pour retrouver un objet Python.

| Python | JSON |
|---|---|
| `dict` | objet `{ }` |
| `list` | liste `[ ]` |
| `str` | chaîne `"..."` |
| `int`, `float` | nombre |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

Pour voir le texte JSON produit, on utilise ici `json.dumps()`. Cette fonction est présentée en détail en partie 3.2.

In [ ]:
notice = {"cote": "MS 24", "date": 1200, "numerise": True, "nb_folios": None}
print(json.dumps(notice))

**Ce qui ne revient pas à l'identique.** On fait l'aller-retour : `json.dumps()` sérialise, puis `json.loads()` désérialise.

Un tuple revient sous forme de **liste** :

In [ ]:
print(json.loads(json.dumps({"datation": (1200, 1210)})))

Une clé numérique revient sous forme de **chaîne** :

In [ ]:
print(json.loads(json.dumps({1200: "MS 24"})))

**Ce qui ne passe pas du tout** : un set, ou un objet `Manuscrit`. Décommentez les lignes pour voir les erreurs.

In [ ]:
# json.dumps({"creatures": {"lion", "dragon"}})
# TypeError: Object of type set is not JSON serializable

# json.dumps(Manuscrit("MS 24", "Aberdeen, University Library", 1200))
# TypeError: Object of type Manuscrit is not JSON serializable

Il faut d'abord convertir l'objet en liste ou en dictionnaire.

#### **Exercice 5 (facultatif) : Rendre un objet `Manuscrit` sérialisable**

1. Créez une classe enfant `ManuscritExportable`, qui hérite de `Manuscrit` et ajoute une méthode `vers_dict()`. Cette méthode renvoie un dictionnaire avec les clés `"cote"`, `"bibliotheque"`, `"date"` et `"creatures"`.
2. Créez l'objet d'Aberdeen, ajoutez-lui la créature `"licorne"`, puis affichez le résultat de `json.dumps()` appliqué à son dictionnaire, avec `ensure_ascii=False`.

*Solution :*

In [ ]:
# Votre code ici

---

## **3. Lire et enregistrer des fichiers**

### **3.1 Ouvrir un fichier : `with open()`**

```python
with open(nom_du_fichier, mode, encoding="utf-8") as fichier:
    ...
```

- `with` ferme le fichier **automatiquement** à la fin du bloc, même en cas d'erreur.
- `encoding="utf-8"` garantit que les accents sont bien lus et écrits, quel que soit le système.

| Mode | Description |
|---|---|
| `"r"` | Lire (`FileNotFoundError` si le fichier n'existe pas) |
| `"w"` | Écrire (crée le fichier, ou **écrase** son contenu) |
| `"a"` | Ajouter à la fin (crée le fichier s'il n'existe pas) |
| `"wb"` | Écrire en binaire (images : séance 7) |

`fichier.write(texte)` écrit une chaîne ; `fichier.read()` lit tout le contenu, sous forme de chaîne.

In [ ]:
with open("colombe.txt", "w", encoding="utf-8") as fichier:
    fichier.write(titre)

with open("colombe.txt", "r", encoding="utf-8") as fichier:
    contenu = fichier.read()

print(contenu)

Le mode `"a"` ajoute à la fin, sans effacer :

In [ ]:
with open("colombe.txt", "a", encoding="utf-8") as fichier:
    fichier.write("\nLes Cyranides")

with open("colombe.txt", "r", encoding="utf-8") as fichier:
    print(fichier.read())

#### **Exercice 6 : Enregistrer la liste des créatures**

1. Écrivez les créatures de la liste `creatures` dans un fichier `creatures.txt`, **une par ligne**.
2. Relisez le fichier et affichez son contenu.
3. Exécutez une seconde fois votre cellule : le fichier contient-il les créatures en double ? Pourquoi ?

*Indice : `"\n"` est le caractère de retour à la ligne.*

*Solution :*

In [ ]:
# Votre code ici

### **3.2 La bibliothèque `json`**

`json` est un module **intégré** à Python (importé dans la cellule de données).

**Avec un fichier**

- `json.dump(objet, fichier)` : enregistre un objet Python dans un fichier JSON.
- `json.load(fichier)` : relit un fichier JSON et renvoie un objet Python.

**Options utiles** (pour `dump` et `dumps`) : `indent=4` présente le fichier avec des retraits ; `ensure_ascii=False` conserve les accents.

In [ ]:
with open("corpus.json", "w", encoding="utf-8") as fichier:
    json.dump(corpus, fichier, ensure_ascii=False, indent=4)

with open("corpus.json", "r", encoding="utf-8") as fichier:
    corpus_relu = json.load(fichier)

print(corpus_relu[0]["cote"])
print(corpus_relu == corpus)

Ouvrez le fichier `corpus.json` depuis l'explorateur de fichiers de Jupyter : vous y retrouvez la syntaxe de la partie 2.2.

**Avec une chaîne de caractères.** Le **`s`** final signifie *string* (chaîne).

- `json.dumps(objet)` : transforme un objet Python en chaîne JSON.
- `json.loads(chaine)` : transforme une chaîne JSON en objet Python.

In [ ]:
notice = json.loads('{"cote": "MS 24", "date": 1200}')
print(notice)
print(type(notice))

L'effet d'`ensure_ascii=False` :

In [ ]:
print(json.dumps({"creatures": ["lion", "panthère"]}))
print(json.dumps({"creatures": ["lion", "panthère"]}, ensure_ascii=False))

*Remarque : on utilise `print()`. Si l'on évalue `json.dumps(...)` seul, Jupyter montre la représentation de la chaîne, avec une barre oblique doublée (`panth\\u00e8re`).*

#### **Exercice 7 : Corriger un JSON mal formé**

La chaîne ci-dessous contient **trois** erreurs de syntaxe JSON.

1. Exécutez la cellule fournie, puis décommentez la ligne `json.loads(...)` : lisez le message d'erreur.
2. Écrivez une chaîne corrigée, `notice_corrigee`, et chargez-la avec `json.loads()`. Corrigez une erreur à la fois, en relisant chaque nouveau message.

*Solution :*

In [ ]:
notice_mal_formee = """{
    'cote': 'MS 24',
    "date": 1200,
    "numerise": True,
}"""

# json.loads(notice_mal_formee)

# Votre code ici

#### **Exercice 8 : Ajouter Berne 318 au corpus**

1. Créez la notice `manuscrit_berne`, avec les mêmes clés que les deux autres : cote `"Cod. 318"`, bibliothèque `"Bern, Burgerbibliothek"`, date `830`, numérisé.
2. Ajoutez-la au `corpus`.
3. Enregistrez le corpus dans `corpus.json`, rechargez-le dans `corpus_relu`, et vérifiez qu'il contient bien **trois** notices.

*Solution :*

In [ ]:
# Votre code ici

### **3.3 Les chemins avec `pathlib`**

`pathlib` est un module **standard** qui représente les chemins de fichiers comme des **objets** (importé dans la cellule de données). Le même code fonctionne sous Windows, macOS et Linux.

- L'opérateur `/` assemble les morceaux avec le bon séparateur.
- `dossier.mkdir(exist_ok=True)` crée le dossier, sans erreur s'il existe déjà.

In [ ]:
dossier = Path("donnees")
dossier.mkdir(exist_ok=True)

chemin = dossier / "ms_24.json"
print(chemin)

Sans `exist_ok=True`, la seconde exécution de la cellule provoquerait `FileExistsError: [Errno 17] File exists: 'donnees'`.

Les propriétés d'un chemin :

In [ ]:
print(chemin.name)
print(chemin.stem)
print(chemin.suffix)
print(chemin.parent)

`read_text()` et `write_text()` lisent et écrivent du texte. Avec `json`, `dumps()` et `loads()` font le lien entre l'objet Python et ce texte :

In [ ]:
print(chemin.exists())

chemin.write_text(json.dumps(manuscrit, ensure_ascii=False, indent=4), encoding="utf-8")
print(chemin.exists())

notice = json.loads(chemin.read_text(encoding="utf-8"))
print(notice)

Un objet `Path` s'utilise aussi avec `open()` : `with open(chemin, "r", encoding="utf-8") as fichier:`.

#### **Exercice 9 : Ranger le corpus dans le dossier `donnees`**

1. Construisez le chemin `donnees/corpus.json` avec `Path` et l'opérateur `/`.
2. Enregistrez-y le corpus avec `write_text()` et `json.dumps()`, puis vérifiez que le fichier existe.
3. Rechargez-le avec `read_text()` et `json.loads()`, et affichez la cote de chaque notice.

*Solution :*

In [ ]:
# Votre code ici

---

## **4. Parcourir un manifeste IIIF**

### **4.1 Un manifeste est un fichier JSON**

Une institution décrit un manuscrit numérisé dans un fichier JSON : le **manifeste**. Nous travaillons sur celui de **Berne, Burgerbibliothek, Cod. 318** (*Physiologus Bernensis*), publié par e-codices.

**Avant de commencer** : placez le fichier `Berne_318.json` fourni dans le dossier `donnees`.

On le lit exactement comme `corpus.json` :

In [ ]:
chemin_manifeste = Path("donnees") / "Berne_318.json"

with open(chemin_manifeste, "r", encoding="utf-8") as fichier:
    manifeste = json.load(fichier)

print(type(manifeste))
print(list(manifeste.keys()))

Les clés qui commencent par `@` viennent du format JSON-LD : nous y reviendrons en séance 7.

### **4.2 Descendre niveau par niveau**

- Un objet (dictionnaire) : on accède à une valeur par sa **clé**, `["label"]`.
- Une liste : on accède à un élément par son **indice**, `[0]`.
- On enchaîne, de gauche à droite, **un niveau à la fois**.

In [ ]:
print(manifeste["label"])

`metadata` est une **liste** de dictionnaires, qui ont chacun une clé `"label"` et une clé `"value"` :

In [ ]:
print(type(manifeste["metadata"]))
print(type(manifeste["metadata"][0]))
print(manifeste["metadata"][0])

In [ ]:
for entree in manifeste["metadata"]:
    print(entree["label"], ":", entree["value"])

### **4.3 Les vues numérisées**

Les images sont rangées dans `sequences` (une liste), dont le premier élément contient `canvases` (une liste de vues). Chaque vue est un dictionnaire.

`manifeste["sequences"][0]["canvases"]` se lit étape par étape : un dictionnaire, puis une liste, puis un dictionnaire.

In [ ]:
canvases = manifeste["sequences"][0]["canvases"]

print(len(canvases))
print(canvases[0]["label"])
print(canvases[4]["label"])

#### **Exercice 10 : Retrouver le titre du manuscrit**

Parcourez `manifeste["metadata"]` avec une boucle `for` et un `if`, et affichez la valeur dont le libellé est `"Title (English)"`.

*Solution :*

In [ ]:
# Votre code ici

#### **Exercice 11 : Trouver la première page du texte**

Avec une boucle `while`, trouvez l'indice de la vue dont le libellé est `"1r"`, comme vous avez cherché « colombe. » dans le titre en séance 1. Affichez les libellés des vues qui la précèdent.

*Solution :*

In [ ]:
# Votre code ici

#### **Exercice 12 : Feuillets et vues**

1. Retrouvez dans `metadata` la valeur de `"Number of Pages"`. De quel type est-elle ?
2. Convertissez-la en nombre, et comparez-la au nombre de vues, `len(canvases)`.
3. Affichez les libellés des six dernières vues. Comment expliquez-vous l'écart ?

*Solution :*

In [ ]:
# Votre code ici

#### **Exercice 13 (facultatif) : La date du manuscrit**

1. Retrouvez dans `metadata` la valeur de `"Date of Origin (English)"`, et donnez-la à `siecle()` dans un `try` / `except`. Que se passe-t-il ?
2. Appelez maintenant `siecle(830)`. Le résultat est-il juste ? Pourquoi ?

Écrivez votre code dans la première cellule, puis répondez dans la cellule de réponse.

*Solution :*

In [ ]:
# Votre code ici

*Votre réponse :*

---

### **Conclusion**

Dans ce notebook, nous avons :

- construit des **décorateurs**, à la main puis avec `@`, et rendus compatibles avec toutes les fonctions grâce à `*args` et `**kwargs` ;
- découvert le format **JSON** et ses correspondances avec les types Python ;
- **lu et enregistré** des fichiers avec `open()`, la bibliothèque `json` et `pathlib` ;
- **parcouru** un manifeste IIIF, le fichier JSON qui décrit un manuscrit numérisé.

À la prochaine séance, nous passerons du tableur à Python avec le format **CSV**.